# IRIS ML Pipeline on Vertex AI — Week 1 (Driver Notebook)

**Roll No:** 23f2004634 · **Branch:** `week_1` · **Term:** MAY 2026

This notebook is a thin driver that runs the pipeline scripts end-to-end so the
whole flow can be demonstrated in one top-to-bottom execution. The real logic
lives in the standalone, reproducible scripts:

- `data_prep.py` — split + upload data to GCS (Task 2)
- `train.py` — fetch data, train, store timestamped artifacts (Task 3)
- `inference.py` — fetch model from GCS, run on eval set (Task 4)


## 0. Environment
Set the project and bucket. These feed `config.py` via environment variables.

In [ ]:
import os
os.environ["PROJECT_ID"] = os.popen("gcloud config get-value project 2>/dev/null").read().strip()
os.environ["BUCKET"] = "23f2004634-mlops-week1"
print("PROJECT_ID:", os.environ["PROJECT_ID"])
print("BUCKET    :", os.environ["BUCKET"])

Confirm the starter data is present (cloned from the `ga_resources` repo, branch `week_1`).

In [ ]:
!ls ga_resources/data/raw ga_resources/data/v1 ga_resources/data/v2

## Task 2 — Store training data in GCS
Stratified train/eval split, uploaded to `gs://<bucket>/data/raw/`.

In [ ]:
!python data_prep.py --src ga_resources/data/raw/iris.csv --version raw

## Task 3 — Train (Run #1)
Fetch data from GCS, train, store artifacts in a **timestamped** folder.

In [ ]:
!python train.py --version raw

## Task 4 — Inference (Run #1)
Fetch the latest model from GCS and run on the eval set.

In [ ]:
!python inference.py --version raw

## Task 5 — Second full run
Produces a **second** timestamped folder, proving traceable, repeatable runs.

In [ ]:
!python train.py --version raw && python inference.py --version raw

### Verify: two timestamped run folders exist for `raw`

In [ ]:
!gsutil ls gs://$BUCKET/artifacts/raw/

## Task 6 (Optional) — Multiple data versions
Run the same pipeline on two different data versions and compare.

In [ ]:
!python data_prep.py --src ga_resources/data/v1/data.csv --version v1
!python train.py --version v1 && python inference.py --version v1

In [ ]:
!python data_prep.py --src ga_resources/data/v2/data.csv --version v2
!python train.py --version v2 && python inference.py --version v2

### Compare eval accuracy across data versions

In [ ]:
import json
from google.cloud import storage
client = storage.Client()
bucket = os.environ["BUCKET"]

print(f"{'version':<8}{'run':<22}{'eval_acc':>9}")
print("-"*40)
for v in ["raw", "v1", "v2"]:
    # latest run for this version
    prefixes = set()
    it = client.list_blobs(bucket, prefix=f"artifacts/{v}/", delimiter="/")
    list(it)
    runs = sorted(p.rstrip('/').split('/')[-1] for p in it.prefixes)
    if not runs:
        continue
    run = runs[-1]
    blob = client.bucket(bucket).blob(f"artifacts/{v}/{run}/inference/eval_metrics.json")
    m = json.loads(blob.download_as_text())
    print(f"{v:<8}{run:<22}{m['eval_accuracy']:>9.3f}")

### Final artifact tree
The full organized output: each version → each timestamped run → model, logs, metrics, and inference outputs.

In [ ]:
!gsutil ls -r gs://$BUCKET/artifacts/ | head -60

---
**Summary:** data fetched from GCS, model trained and stored as timestamped
artifacts, inference run from GCS-fetched models, repeated twice, and compared
across data versions — all on GCP / Vertex AI Workbench.